# IOS Risk — Project 03 Evaluation

Scores the fine-tuned adapter against Project 03's success criteria and against
the Project 01 XGBoost baseline.

**Targets:** tier accuracy > 0.70 · avg quality > 0.60
**Baseline:** XGBoost — P 0.9011 · R 0.8367 · F1 0.8677

Runs BOTH the base model and the tuned adapter over the same 250 held-out cases,
so the comparison is like-for-like. About 20 minutes on a T4.

## Before running — two inputs are required

**Add Input -> Datasets ->** `ios-risk-eval-assets` (the test set and scoring code)
**Add Input -> Notebooks ->** the training run, **v2 version**

**Settings -> Accelerator -> GPU T4 x2**

This notebook makes no network calls beyond downloading the base model. It does
not clone anything: an earlier version cloned a private repo, git blocked on a
credential prompt with no stdin, and the session burned 12 hours before Kaggle
killed it.

### Step 1: GPU check — fail in seconds, not hours

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU: {name}  sm_{major}{minor}  ({torch.cuda.device_count()} visible)")

# Kaggle's API cannot set the accelerator, so an API-pushed notebook lands on a
# P100 (sm_60) which Unsloth cannot use. Stop here rather than hang on load.
assert major >= 7, (
    f"{name} is sm_{major}{minor}; sm_70+ required.\n"
    "Fix: Settings -> Accelerator -> 'GPU T4 x2', then re-run."
)
print("GPU OK.")

### Step 2: Locate both inputs

In [ ]:
import os, sys

def find(pred, label):
    hits = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        if pred(root, files):
            hits.append(root)
    if not hits:
        print(f"--- {label}: NOT FOUND. /kaggle/input contains:")
        for r, _d, _f in os.walk("/kaggle/input"):
            if r.rstrip("/").count("/") - 2 <= 4:
                print("   ", r)
        raise AssertionError(f"{label} not attached.")
    return hits

# Eval assets: the dataset holding testset.json and domain_eval.py
assets = find(lambda r, f: "testset.json" in f and "domain_eval.py" in f,
              "eval assets dataset")[0]
sys.path.insert(0, assets)
print("eval assets :", assets)

# Adapter: a directory holding adapter_config.json, excluding intermediate
# checkpoints. Must be the v2 adapter — v1 was trained on the broken dataset.
adapters = [r for r in find(lambda r, f: "adapter_config.json" in f, "trained adapter")
            if "checkpoint" not in r]
v2 = [a for a in adapters if "v2" in a]
if not v2:
    raise AssertionError(
        f"No v2 adapter among {adapters}.\n"
        "The attached notebook version is wrong — attach the v2 training run."
    )
ADAPTER = v2[0]
print("adapter     :", ADAPTER)

### Step 3: Install (same pins as training)

In [ ]:
!pip install -q \
    "transformers==5.5.0" \
    "datasets==4.3.0" \
    "trl==0.24.0" \
    "bitsandbytes==0.50.1" \
    "xformers==0.0.34" \
    "peft>=0.18.0" \
    unsloth unsloth_zoo 2>&1 | tail -6
print("install done")

### Step 4: Score the BASE model

The control. Any score for the fine-tune is meaningless without it.

In [ ]:
import unsloth  # must precede transformers/trl
from domain_eval import run

base_summary = run(
    model_id="unsloth/Meta-Llama-3.1-8B-Instruct",
    tag="base",
    testset_path=os.path.join(assets, "testset.json"),
    out_dir="/kaggle/working/eval_results",
)

### Step 5: Score the TUNED adapter

In [ ]:
tuned_summary = run(
    model_id=ADAPTER,
    tag="tuned",
    testset_path=os.path.join(assets, "testset.json"),
    out_dir="/kaggle/working/eval_results",
)

### Step 6: Verdict

In [ ]:
import json

def row(label, b, t, target=None):
    flag = ""
    if target is not None:
        flag = "   PASS" if t > target else "   FAIL"
    print(f"{label:<18} base {b:>7.4f}   tuned {t:>7.4f}   delta {t-b:>+7.4f}{flag}")

ra_b, ra_t = base_summary["risk_assessment"], tuned_summary["risk_assessment"]
print("RISK ASSESSMENT  (50 held-out scenario prompts)")
row("tier accuracy",  ra_b["tier_accuracy"],  ra_t["tier_accuracy"],  0.70)
row("avg quality",    ra_b["avg_quality"],    ra_t["avg_quality"],    0.60)
row("reasoning rate", ra_b["reasoning_rate"], ra_t["reasoning_rate"])
row("action rate",    ra_b["action_rate"],    ra_t["action_rate"])

cb, ct = base_summary["classification"], tuned_summary["classification"]
print("\nCLASSIFICATION  (200 balanced held-out transactions)")
row("precision", cb["precision"], ct["precision"])
row("recall",    cb["recall"],    ct["recall"])
row("f1",        cb["f1"],        ct["f1"])
print(f"{'':<18} XGBoost (Project 01):  P 0.9011  R 0.8367  F1 0.8677")
print(f"unparseable — base {cb['unparseable_responses']}, tuned {ct['unparseable_responses']}")

print("\nPROJECT 03 CRITERIA:", "PASS" if tuned_summary["passes_project03"] else "FAIL")

with open("/kaggle/working/eval_results/comparison.json", "w") as f:
    json.dump({"base": base_summary, "tuned": tuned_summary}, f, indent=2)
print("\nwritten -> /kaggle/working/eval_results/")

### Step 7: Read actual outputs

Metrics hide behaviour. The v2 smoke test scored correctly on all three probes
but emitted `Probability of fraud: 89.4%` and `Total score: 68` — numbers with no
basis in the training data or the input. Aggregates do not surface that.

In [ ]:
import json, re

rows = json.load(open("/kaggle/working/eval_results/eval_tuned.json"))["risk_rows"]

for r in rows[:5]:
    mark = "OK  " if r["correct_tier"] else "MISS"
    print(f"[{mark}] expected {r['expected_tier']:<8} got {str(r['predicted_tier']):<8} ({r['expected_pattern']})")
    print("   ", r["input"])
    print("   ", r["response"][:300].replace("\n", " "))
    print()

suspect = [r for r in rows if re.search(r"\d+(\.\d+)?\s*%|score:\s*\d+", r["response"], re.I)]
print(f"responses containing an invented probability or score: {len(suspect)}/{len(rows)}")
for r in suspect[:3]:
    m = re.search(r"[^.]*(?:\d+(?:\.\d+)?\s*%|score:\s*\d+)[^.]*\.", r["response"], re.I)
    if m:
        print("   ", m.group(0).strip()[:160])